# 📘 Notebook 11: Maximum Likelihood Estimation (MLE)

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐⭐ (Advanced)

---

## 🏠 Why Does This Matter? (The Grand Unification)

You've learned:
- Calculus → how to find the minimum of a function
- Gradient descent → the algorithm to minimize
- Probability distributions → what shapes data can take

Now: **why do we use MSE loss for house price regression?** Why not MAE, or something else?

**Answer: MLE.**

MLE says: "Find the parameters that make your observed house prices **most probable** under your model."

When you assume Gaussian noise → MLE gives you MSE.  
When you assume Bernoulli output → MLE gives you binary cross-entropy.  
When you add a Gaussian prior on weights → MAP gives you MSE + L2 regularization.

**Every standard loss function in ML has a probabilistic derivation via MLE.**

---

## 📚 Prerequisites
- All previous notebooks (especially 8–10)

---

## Part 1: The MLE Framework

### Plain English First

You observe house prices: $320k, $410k, $280k, $350k, ...  
You assume they follow a Gaussian distribution with unknown mean `μ` and variance `σ²`.

**Question:** What values of `μ` and `σ²` make these observations most probable?

**Setup:**
1. Write the **likelihood**: $L(\theta) = P(\text{data}; \theta)$ — how probable is the data under parameters θ?
2. Assume data points are **independent**: $L(\theta) = \prod_{i=1}^{N} P(x_i; \theta)$
3. **Maximize** the likelihood by taking derivatives and setting to 0

**Log-likelihood trick:** Products become sums under log.  
Since log is monotone, maximizing `log L(θ)` = maximizing `L(θ)`:

$$\hat{\theta}_{MLE} = \arg\max_\theta \ell(\theta) = \arg\max_\theta \sum_{i=1}^{N} \log P(x_i; \theta)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# ─────────────────────────────────────────────────────────────────
# MLE EXAMPLE 1: Finding the best Gaussian to fit house prices
#
# Data: 50 house prices (log-transformed to look approximately Gaussian)
# Parameters: μ (mean log-price), σ (std log-price)
# Analytical MLE: μ̂ = sample mean, σ̂ = sample std (divide by N)
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
true_mu_log    = np.log(350_000)   # true mean of log-prices
true_sigma_log = 0.4               # true std of log-prices

# Simulate 50 observed house sales
n_samples    = 50
log_prices   = np.random.normal(true_mu_log, true_sigma_log, n_samples)
obs_prices   = np.exp(log_prices)   # actual prices in dollars

def log_likelihood(mu, sigma, data):
    """
    Compute the log-likelihood of the data under Gaussian(mu, sigma).

    For each data point x_i:
        log P(x_i; mu, sigma) = -log(sigma) - 0.5*log(2π) - (x_i - mu)²/(2σ²)

    Total log-likelihood = sum over all data points.
    """
    if sigma <= 0: return -np.inf
    n = len(data)
    # Sum of log-probabilities for each observation
    return np.sum(norm.logpdf(data, loc=mu, scale=sigma))

# ─── MLE estimates (analytical closed-form solutions) ─────────────
# Derived by taking d(log L)/dμ = 0 and d(log L)/dσ = 0:
mu_mle    = np.mean(log_prices)           # sample mean
sigma_mle = np.std(log_prices, ddof=0)    # sample std (divide by N, not N-1)

print("MLE for Gaussian: Fitting a Distribution to House Price Data")
print(f"  N = {n_samples} observed house sales")
print()
print(f"  True parameters:     μ = {true_mu_log:.4f}, σ = {true_sigma_log:.4f}")
print(f"  MLE estimates:       μ̂ = {mu_mle:.4f}, σ̂ = {sigma_mle:.4f}")
print()
print(f"  Log-likelihood at true params: {log_likelihood(true_mu_log, true_sigma_log, log_prices):.4f}")
print(f"  Log-likelihood at MLE:         {log_likelihood(mu_mle, sigma_mle, log_prices):.4f}")
print(f"  MLE has higher (or equal) log-likelihood? {log_likelihood(mu_mle,sigma_mle,log_prices) >= log_likelihood(true_mu_log,true_sigma_log,log_prices)}  ✓")
print()
print(f"  MLE estimated median price: ${np.exp(mu_mle):,.0f}")
print(f"  True median price:          ${np.exp(true_mu_log):,.0f}")

# ─── Plot: likelihood surface and maximum ─────────────────────────
mu_range    = np.linspace(true_mu_log - 0.5, true_mu_log + 0.5, 100)
sigma_range = np.linspace(0.1, 0.8, 100)
Mu, Sig     = np.meshgrid(mu_range, sigma_range)

# Compute log-likelihood at each (μ, σ) grid point
LL = np.array([[log_likelihood(m, s, log_prices) for m in mu_range]
               for s in sigma_range])

plt.figure(figsize=(10, 5))
plt.contourf(Mu, Sig, LL, levels=30, cmap='viridis')
plt.colorbar(label='Log-likelihood ℓ(μ, σ)')
plt.plot(mu_mle, sigma_mle, 'r*', markersize=18, label=f'MLE: μ̂={mu_mle:.3f}, σ̂={sigma_mle:.3f}', zorder=10)
plt.plot(true_mu_log, true_sigma_log, 'w^', markersize=12, label=f'True: μ={true_mu_log:.3f}, σ={true_sigma_log:.3f}', zorder=10)
plt.xlabel('μ (mean log-price)'); plt.ylabel('σ (std log-price)')
plt.title('Log-Likelihood Surface: MLE Finds the Maximum (Red Star)', fontsize=12)
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Part 2: MLE → MSE Loss (The Key Derivation!)

### Plain English First

Your house price model predicts `f(x; w)` (a function of features x and weights w).  
The actual price `y` is the prediction plus some **random noise**: `y = f(x; w) + ε`.

If we assume the noise is **Gaussian**: `ε ∼ N(0, σ²)`, then:
$$y | x \sim \mathcal{N}(f(x; w), \sigma^2)$$

The likelihood of one observation is:
$$P(y_i | x_i; w) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(y_i - f(x_i; w))^2}{2\sigma^2}\right)$$

Total log-likelihood over N houses:
$$\ell(w) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{N}(y_i - f(x_i; w))^2$$

**Maximizing `ℓ(w)` = minimizing `Σ(y_i - f(x_i; w))²` = minimizing MSE!**  
The constant terms disappear, and maximizing the negative = minimizing the positive.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DERIVATION: MLE for linear regression = minimizing MSE
#
# Model: price = w1*sqft + w2*beds + ε,  ε ~ N(0, σ²)
# Show that maximizing log-likelihood and minimizing MSE
# give the SAME optimal weights.
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N = 200

# Generate house data
X_data = np.random.randn(N, 2)              # (sqft_norm, beds_norm)
w_true = np.array([2.0, -1.0])             # true weights
sigma  = 0.5                               # noise level
y_data = X_data @ w_true + sigma * np.random.randn(N)   # prices with Gaussian noise

def mse_loss(w):
    """MSE loss: (1/N) * sum((Xw - y)²)"""
    residuals = X_data @ w - y_data
    return np.mean(residuals**2)

def neg_log_likelihood(w):
    """
    Negative log-likelihood under Gaussian noise assumption.
    NLL = (N/2)*log(2π) + (N/2)*log(σ²) + (1/(2σ²)) * sum((yi - f(xi;w))²)

    The σ-dependent terms are constant w.r.t. w, so minimizing NLL
    is equivalent to minimizing sum((yi - f(xi;w))²) = N*MSE!
    """
    residuals = X_data @ w - y_data
    sse       = np.sum(residuals**2)   # sum of squared errors
    # Full NLL including all terms
    nll = (N/2) * np.log(2*np.pi*sigma**2) + sse / (2*sigma**2)
    return nll

# ─── Find optimal weights by gradient descent on both objectives ──
def gradient_mse(w):
    return 2 * X_data.T @ (X_data @ w - y_data) / N

def gradient_nll(w):
    residuals = X_data @ w - y_data
    return X_data.T @ residuals / sigma**2   # derivative of NLL w.r.t. w

# Run gradient descent from same start for both
w_mse = np.zeros(2)
w_nll = np.zeros(2)

for _ in range(500):
    w_mse -= 0.05 * gradient_mse(w_mse)
    w_nll -= 0.00025 * gradient_nll(w_nll)  # different lr due to different scale

print("MLE = MSE: Same Optimal Weights!")
print(f"  True weights:            {w_true}")
print(f"  MSE minimizer:           {w_mse.round(4)}")
print(f"  NLL minimizer (MLE):     {w_nll.round(4)}")
print()
print(f"  MSE at MSE-optimal w:    {mse_loss(w_mse):.6f}")
print(f"  MSE at NLL-optimal w:    {mse_loss(w_nll):.6f}")
print()
print("Both methods converge to the same weights — MSE IS the MLE objective for Gaussian noise!")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MLE for binary classification → binary cross-entropy loss
#
# Model: P(y=1 | x; w) = sigmoid(w.T @ x) = 1/(1+e^{-w.T x})
# Assumption: y | x ~ Bernoulli(sigmoid(w.T x))
#
# Log-likelihood: ℓ(w) = Σ [y·log(ŷ) + (1-y)·log(1-ŷ)]
# Negative: -ℓ = binary cross-entropy
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N_cls = 200

# Generate classification data: predict if house is expensive
X_cls    = np.random.randn(N_cls, 2)              # features
w_cls_true = np.array([1.5, -1.0])               # true weights
logits   = X_cls @ w_cls_true                    # raw scores
y_cls    = (np.random.rand(N_cls) < 1/(1+np.exp(-logits))).astype(float)   # binary labels

def sigmoid(z):
    """Sigmoid function: maps any real number to (0, 1)."""
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))   # clip prevents overflow

def binary_cross_entropy(w):
    """
    Negative log-likelihood for Bernoulli output.

    = -sum [y*log(σ(w.T x)) + (1-y)*log(1-σ(w.T x))] / N

    This IS binary cross-entropy (what PyTorch calls nn.BCELoss).
    Minimizing BCE = maximizing Bernoulli likelihood.
    """
    y_hat = sigmoid(X_cls @ w)            # predicted probabilities
    eps   = 1e-10                         # tiny value to avoid log(0)
    return -np.mean(y_cls * np.log(y_hat + eps) +
                    (1 - y_cls) * np.log(1 - y_hat + eps))

def bce_gradient(w):
    """Gradient of binary cross-entropy w.r.t. w."""
    y_hat     = sigmoid(X_cls @ w)
    residuals = y_hat - y_cls             # prediction error
    return X_cls.T @ residuals / N_cls

# Train with gradient descent
w_cls = np.zeros(2)
bce_history = []

for step in range(500):
    bce_history.append(binary_cross_entropy(w_cls))
    w_cls -= 0.5 * bce_gradient(w_cls)

# ─── Plot training curve ──────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(bce_history, 'steelblue', linewidth=2)
plt.xlabel('Training step'); plt.ylabel('Binary Cross-Entropy (= Negative Log-Likelihood)')
plt.title('Training a Classifier: Minimizing Cross-Entropy = Maximizing Bernoulli Likelihood',
          fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"True weights:    {w_cls_true}")
print(f"Learned weights: {w_cls.round(3)}")
print()

# Check accuracy
y_pred_probs = sigmoid(X_cls @ w_cls)
y_pred_class = (y_pred_probs > 0.5).astype(float)
accuracy     = np.mean(y_pred_class == y_cls)
print(f"Classification accuracy: {accuracy*100:.1f}%")

## Part 3: The Grand Unification

Every loss function you'll use in ML is a negative log-likelihood under some distribution assumption:

In [ ]:
# ─────────────────────────────────────────────────────────────────
# The Grand Unification: Loss function = -log-likelihood
# ─────────────────────────────────────────────────────────────────

print("THE GRAND UNIFICATION OF ML LOSS FUNCTIONS")
print("=" * 70)
print()
print("Every standard loss function is a negative log-likelihood under some distribution:")
print()

unified = [
    ("MSE (Mean Squared Error)",
     "y | x ~ Gaussian(f(x;w), σ²)",
     "Regression (house price)",
     "-log P = const + (y - ŷ)²/(2σ²)  →  minimize (y - ŷ)²"),

    ("Binary Cross-Entropy",
     "y | x ~ Bernoulli(σ(f(x;w)))",
     "Binary classification (expensive?)",
     "-log P = -[y·log(ŷ) + (1-y)·log(1-ŷ)]"),

    ("Categorical Cross-Entropy",
     "y | x ~ Categorical(softmax(f(x;w)))",
     "Multi-class (cheap/medium/expensive)",
     "-log P = -log(ŷ[y])  (neg log of correct class prob)"),

    ("MAE (Mean Absolute Error)",
     "y | x ~ Laplace(f(x;w), b)",
     "Robust regression (outlier houses)",
     "-log P = const + |y - ŷ|/b  →  minimize |y - ŷ|"),

    ("MSE + L2 regularization",
     "y|x ~ Gaussian, w ~ Gaussian(0,τ²)",
     "Ridge regression (prevent overfitting)",
     "-log P(y|x;w) - log P(w) = MSE + λ‖w‖²"),
]

for loss_name, distribution, use_case, derivation in unified:
    print(f"  {'━'*65}")
    print(f"  Loss:         {loss_name}")
    print(f"  Assumes:      {distribution}")
    print(f"  ML task:      {use_case}")
    print(f"  Connection:   {derivation}")

print(f"  {'━'*65}")
print()
print("Key message: When you choose a loss function, you are making a probabilistic")
print("assumption about how your data is distributed. Choose wisely!")

---

## ✅ Self-Check Questions

**1.** You're predicting house prices and switch from MSE to MAE loss.  
   What distributional assumption changed? When is MAE better than MSE?

**2.** Write the log-likelihood for 3 independent Gaussian observations x₁=2, x₂=3, x₃=4,  
   under N(μ=3, σ=1). Then compute it numerically.

**3.** The binary cross-entropy loss `−[y·log(ŷ) + (1−y)·log(1−ŷ)]`:  
   What happens to this loss when `ŷ = y_true` (perfect prediction)?  
   What happens when `ŷ = 0.5` (maximum uncertainty)?

**4.** You add L2 regularization: `Loss = MSE + λ‖w‖²`.  
   What probabilistic prior on `w` does this correspond to?  
   What does large `λ` mean about your prior belief?

**5.** MLE is a special case of MAP with a specific prior. What prior?  
   (Hint: which prior would not change the MLE objective at all?)

<details>
<summary>Click to see answers</summary>

1. MSE assumes **Gaussian noise**; MAE assumes **Laplace noise**. Laplace has heavier tails, so it is less sensitive to outliers. MAE is better when luxury homes (extreme prices) shouldn't dominate the loss.

2. `log L = log N(2;3,1) + log N(3;3,1) + log N(4;3,1)`. Each term: `−log(√2π) − (xᵢ−3)²/2`. Sum = `3×(−log(√2π)) − (1+0+1)/2 = −3×0.919 − 1 = −4.756`.

3. When `ŷ = y_true` (e.g., `ŷ=1, y=1`): loss = `−[1×log(1) + 0×log(0)] = 0` (perfect, no loss). When `ŷ=0.5`: loss = `−[y×log(0.5) + (1−y)×log(0.5)] = log(2) ≈ 0.693` regardless of y.

4. L2 regularization corresponds to a **Gaussian prior `w ∼ N(0, 1/(2λ))`** on the weights. Large `λ` = small prior variance = strong belief that weights should be near 0.

5. MLE = MAP with a **uniform prior** P(θ) = constant. A uniform prior doesn't change the posterior ranking (multiplying by a constant doesn't change which θ is maximum), so MAP = MLE.

</details>

---

## 📌 Summary

| Concept | Meaning | House price example |
|---------|---------|--------------------|
| **Likelihood L(θ)** | P(data; θ) — how probable is the data? | How likely are these prices given our model? |
| **Log-likelihood ℓ** | Σ log P(xᵢ; θ) — sum not product | Stable to compute for large datasets |
| **MLE** | argmax ℓ(θ) | Best weights that explain the observed prices |
| **Gaussian → MSE** | Assume Gaussian noise → minimize MSE | Standard regression assumption |
| **Bernoulli → BCE** | Assume Bernoulli output → minimize cross-entropy | Binary classifier training |
| **MAP = MLE + prior** | Add log P(w) as regularization | L2 reg = Gaussian prior on weights |

**➡️ Next Notebook:** Hands-On Lab — putting it all together with a full gradient descent visualizer and MSE gradient verification.